# Testes qui-quadrado, pareados e Kolmogorov-Smirnov



---



Relevância para Data Science

#### Seleção de Atributos (*Feature Selection*)

O teste **Qui-Quadrado** é fundamental para identificar se variáveis categóricas são independentes do alvo (*target*).

Se forem independentes, essas variáveis podem ser removidas para simplificar o modelo.

#### Validação de Suposições

O teste **KS (Kolmogorov-Smirnov)** é amplamente usado para verificar se os resíduos de um modelo seguem uma distribuição normal.

Essa é uma premissa importante para muitos algoritmos de regressão e testes estatísticos paramétricos.

#### Análise de Experimentos (Testes A/B)

Testes pareados permitem avaliar se uma nova funcionalidade ou tratamento gerou uma mudança real no comportamento do mesmo usuário.

Isso ajuda a isolar fatores externos e analisar melhor o impacto da mudança.

---

## Teste qui-quadrado

Quando usar:

Variáveis categóricas (categoria vs categoria)

Dados são categorias e há tabelas (contagens)


* No teste qui-quadrado,  podemos comparar distribuições e verificar se valores obtidos estão de acordo com o esperado.
* Por exemplo,  podemos verificar se a saída de um dado, após um grande número de lançamentos,  corresponde à saída de um dado justo.   
* Para realizar o teste qui-quadrado,  precisamos calcular a seguinte métrica:
$$
    \chi^2 = \sum_{i=1}^s \frac{(O_i-E_i)^2}{E_i},
$$    
onde $O_i$ é o valor observado e $E_i$ é o valor esperado.



---



**Exemplo:**  Suponha que lançamos um dados 60 vezes, obtendo valores abaixo. É possível dizer que o dado é justo ao nível de 5\%?

Vamos definir as seguintes hipóteses:
$$
\begin{aligned}
&H_0: p_i = 1/6,  \quad i=1,2,\ldots, 6.\\
&H_1: p_i \neq 1/6 \quad \text{para ao menos um valor }i.
\end{aligned}
$$
Portanto, assumimos como hipótese nula que o dado é justo.  Na hipótese alternativa, se ao menos um dos valores da face for mais provável do que os demais, temos que o dado não é honesto.

In [ ]:
from scipy import stats

def chi2(O,E):
    c = 0
    for i in range(len(O)):
        c = c + ((O[i]-E[i])**2)/E[i]
    return c

obs = [11,7,9,15,12,6]
esp = [10,10,10,10,10,10]

c = chi2(obs,esp)
k = len(obs)-1
alpha = 1 - stats.chi2.cdf(c,k)
print('valor p = ', alpha)

valor p =  0.3471050682817154


Como o valor p é maior do que 0,05, nós não podemos rejeitar a hipótese nula. Logo, não há evidências de que o dado não é justo.




---



**Exemplo:** Em um estudo com 395 participantes,  pesquisadores coletaram dados sobre escolaridade de funcionários em dois setores de uma empresa de tecnologia.  Os dados são mostrados na tabela a seguir.

| Setor | Fundamental | Médio | Técnico | Graduação | Total
| --- | --- | --- | --- | --- | --- |
| Setor 1 | 60 | 54 | 46 | 41 | 201 |
| Setor 2 | 40 | 44 | 53 | 57 | 194 |
| Total | 100 | 98 | 99 | 98 | 395 |

Podemos afirmar que há diferença entre o nível educacional nos dois setores da empresa?

In [ ]:
from scipy.stats import chi2_contingency
import numpy as np

print('Tabela de contigência:')
T = np.array([[60, 54, 46, 41],
              [40, 44, 53, 57]])
print(T)
# realiza o teste chi-quadrado
stat, p, dof, E = chi2_contingency(T)
print('Graus de liberdade=%d' % dof)
print('Frequência esperada:')
print(E)
chi2 = 0
for i in range(0, T.shape[0]):
    for j in range(0, T.shape[1]):
        chi2=chi2 + ((T[i,j]-E[i,j])**2)/(E[i,j])
print('Chi squared: ', chi2)
print('Valor p:', p)

Tabela de contigência:
[[60 54 46 41]
 [40 44 53 57]]
Graus de liberdade=3
Frequência esperada:
[[50.88607595 49.86835443 50.37721519 49.86835443]
 [49.11392405 48.13164557 48.62278481 48.13164557]]
Chi squared:  8.006066246262538
Valor p: 0.045886500891747214


Como o valor p é menor que o nível de significância (e.g., 0.05), rejeitamos a hipótese nula. Neste caso, a hipótese nula é que não há diferença entre os níveis educacionais nos dois setores.



---



**Exemplo:** Em uma universidade,  coleta-se dados de disciplinas optativas, relacionadas à ciências, matemática e arte,  que alunos de engenharia e letras se matriculam, conforme a tabela abaixo.  Calcule o valor p e verifique se a escolha das disciplinas depende do curso do aluno.

| Curso | Ciências | Matemática | Arte
| --- | --- | --- | --- |
| Engenharia | 20 | 30 | 15
| Letras | 20 | 15 | 30

In [ ]:
# chi-squared test with similar proportions
from scipy.stats import chi2_contingency
import numpy as np

print('Tabela de contigência:')
T = np.array([[20,30,15],[20,15,30]])
print(T)
# realiza o teste chi-quadrado
stat, p, dof, E = chi2_contingency(T)
print('Graus de liberdade=%d' % dof)
print('Frequência esperada:')
print(E)
chi2 = 0
for i in range(0, T.shape[0]):
    for j in range(0, T.shape[1]):
        chi2=chi2 + ((T[i,j]-E[i,j])**2)/(E[i,j])
print('Chi squared: ', chi2)
print('Valor p:', p)

Tabela de contigência:
[[20 30 15]
 [20 15 30]]
Graus de liberdade=2
Frequência esperada:
[[20.  22.5 22.5]
 [20.  22.5 22.5]]
Chi squared:  10.0
Valor p: 0.006737946999085468


## Teste pareado

**Exemplo:** Uma empresa farmacêutica desenvolve uma nova droga para reduzir o nível de triglicerídeos no sangue.  São coletados dados de 10 pacientes, antes e depois da administração da droga, conforme a tabela a seguir. Verifique se a droga é efetiva.

| Paciente | 1 | 2 | 3 |4 | 5| 6| 7| 8| 9| 10
| --- | --- | --- | --- |--- |--- |--- |--- |--- |--- |--- |
| Antes | 250| 180| 200| 190| 200| 210| 250| 200| 200| 160
| Depois | 220| 190| 160| 180| 210| 180| 170| 150| 150| 140

In [ ]:
import numpy as np
import scipy.stats
# Dados
Xa = [250, 180, 200, 190, 200, 210, 250, 200, 200, 160]
Xd = [220, 190, 160, 180, 210, 180, 170, 150, 150, 140 ]

# Hipóteses:
# H0: A droga não é efetiva (média das diferenças = 0)
# H1: A droga é efetiva (média das diferenças > 0)  - Teste unilateral

print('Média antes:', np.mean(Xa))
print('Média depois:', np.mean(Xd))
# calcula a diferença
d = 0
n = len(Xa)
for i in range(0, n):
    d = d + Xa[i]-Xd[i]
d = d/n
print('D médio:', d)
# calcula o desvio padrão amostral
sd = 0
for i in range(0,n):
    Di = Xa[i]-Xd[i]
    sd = sd + (Di-d)**2
sd = np.sqrt(sd/(n-1))
print('sd:', sd)
# estatística T
mu = 0
T = (d-mu)/(sd/np.sqrt(n))
print('T:', T)
# valor-p
valor_p = 1 - scipy.stats.t.cdf(T, n-1)
print('valor-p =', valor_p)

if valor_p < 0.05:
    print("Rejeitamos a hipótese nula. Há evidências de que a droga é efetiva.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para afirmar que a droga é efetiva.")

Média antes: 204.0
Média depois: 175.0
D médio: 29.0
sd: 28.06737924669451
T: 3.2673535829207565
valor-p = 0.0048618781914541165
Rejeitamos a hipótese nula. Há evidências de que a droga é efetiva.




---



**Exemplo:** Um empresa realiza um treinamento em oito funcionários a fim de diminuir o tempo de montagem de um motor elétrico, medido em horas.  Foram coletados os tempos para montar o motor antes e depois do treinamento, conforme a tabela  a seguir.

| Funcionário | 1 | 2 | 3 |4 | 5| 6| 7| 8
| --- | --- | --- | --- |--- |--- |--- |--- |---
| Antes | 8| 9| 7| 8| 9| 9| 7| 8
| Depois |7 | 9| 9| 8| 7| 8| 7| 8

Realize um teste de hipóteses para verificar se o treinamento é efetivo na redução do tempo de montagem dos motores.

In [ ]:
import numpy as np
import scipy.stats
# Dados
Xa = [8, 9, 7, 8, 9, 9, 7, 8]
Xd = [7, 9, 9, 8, 7, 8, 7, 8]

# Hipóteses:
# H0: O treinamento não é efetivo (média das diferenças = 0)
# H1: O treinamento é efetivo (média das diferenças < 0) - Teste unilateral

print('Média antes:', np.mean(Xa))
print('Média depois:', np.mean(Xd))
# calcula a diferença
d = 0
n = len(Xa)
for i in range(0, n):
    d = d + Xa[i]-Xd[i]
d = d/n
print('D médio:', d)
# calcula o desvio padrão amostral
sd = 0
for i in range(0,n):
    Di = Xa[i]-Xd[i]
    sd = sd + (Di-d)**2
sd = sd/(n-1)
sd = np.sqrt(sd)
print('sd:', sd)
# estatística T
mu = 0
T = (d-mu)/(sd/np.sqrt(n))
print('T:', T)
# valor-p
valor_p = scipy.stats.t.cdf(-T, n-1)
print('valor-p =', valor_p)

if valor_p < 0.05:
    print("Rejeitamos a hipótese nula. Há evidências de que o treinamento é efetivo na redução do tempo de montagem.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para afirmar que o treinamento é efetivo.")

Média antes: 8.125
Média depois: 7.875
D médio: 0.25
sd: 1.164964745021435
T: 0.6069769786668839
valor-p = 0.28151390184927005
Não rejeitamos a hipótese nula. Não há evidências suficientes para afirmar que o treinamento é efetivo.


In [ ]:
from scipy import stats
import pandas as pd
from sklearn import datasets

X = [8, 9, 7, 8, 9, 9, 7, 8]
Y = [7, 9, 9, 8, 7, 8, 7, 8]

# Realiza o teste t pareado
t_statistic, p_value = stats.ttest_rel(X, Y)
p_value = p_value/2 # dividimos por dois porque a biblioteca faz um teste bilateral
print(f"Estatística t: {t_statistic}")
print(f"Valor-p: {p_value}")

if p_value < 0.05:
    print("Rejeitamos a hipótese nula. Há evidências de que o treinamento é efetivo na redução do tempo de montagem.")
else:
    print("Não rejeitamos a hipótese nula. Não há evidências suficientes para afirmar que o treinamento é efetivo.")

Estatística t: 0.6069769786668839
Valor-p: 0.28151390184927005
Não rejeitamos a hipótese nula. Não há evidências suficientes para afirmar que o treinamento é efetivo.




---



Podemos fazer um teste para comparar a média de duas populações. Nesse caso, as observações são independentes.
* $H_0: \mu_1 = \mu_2$
* $H_a: \mu_1 \neq \mu_2$

In [ ]:
import numpy as np
from scipy.stats import ttest_ind

# Example data
group1 = np.random.normal(10, 2, 30)
group2 = np.random.normal(12, 2, 30)

# Two-sample t-test
t_stat, p_value = ttest_ind(group1, group2, equal_var=False)
print('Valor p:', p_value)

Valor p: 9.511858775498419e-05


In [ ]:
import scipy.stats as stats
import numpy as np


# Dados
data_group1 = np.array([14, 15, 15, 16, 13, 8, 14,
                        17, 16, 14, 19, 20, 21, 15,
                        15, 16, 16, 13, 14, 12])

data_group2 = np.array([15, 17, 14, 17, 14, 8, 12,
                        19, 19, 14, 17, 22, 24, 16,
                        13, 16, 13, 18, 15, 13])

# Perform the two sample t-test with equal variances
t_stat, p_value = stats.ttest_ind(a=data_group1, b=data_group2, equal_var=False)
print('Valor p:', p_value)

Valor p: 0.5302413334606599




---



## Teste de Kolmogorov-Smirnov

Vamos verificar se um conjunto de dados foram gerados por uma distribuição uniforme.
* $H_0:$ Dados vieram de um uniforme.
* $H_a:$ Dados não vieram de um uniforme.

In [ ]:
import numpy as np
from scipy.stats import kstest

# amostra
data = np.array([0.55, 0.35, 0.66, 0.15, 0.44,
                 0.08,0.22, 0.93, 0.19, 0.54])

# Distribuição teórica a ser testada
dist = 'uniform'

# Realizando o teste KS
KS = kstest(data, dist)

# Imprimindo o resultado
print('Valor p:', KS.pvalue)

Valor p: 0.4841115325000003


Como p > 0.05, há evidência de que os dados são de uma distribuição uniforme.

Vamos gerar os dados a partir de uma normal e verificar se o valor p será alto, quando comparamos com uma normal.

In [ ]:
import numpy as np
from scipy.stats import norm, kstest
np.random.seed(2011)

# gera os dados
mean = 5; std = 10
n = 100
data = np.random.normal(mean,std,n)
#data = np.array([0.55, 0.35, 0.66, 0.15, 0.44,0.08,0.22, 0.93, 0.19, 0.54])
#data = np.random.exponential(mean, n)
# define a distribuição teórica a ser testada
dist = 'norm'
# realiza o teste KS
KS = kstest(data, dist, args=(mean, std))
print('Valor p:',KS.pvalue)

Valor p: 0.41047753604311865


In [ ]:
import pandas as pd
from sklearn import datasets

# carrega os dados
iris = datasets.load_iris()
# transforma em um dataframe do Pandas
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
print('Variáveis:', df.columns)
df.head()

Variáveis: Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


**Exemplo:** Vamos verificar se os dados da flor iris são normalmente distribuídos.

In [ ]:
import pandas as pd
from sklearn import datasets

# carrega os dados
iris = datasets.load_iris()
# transforma em um dataframe do Pandas
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
print('Variáveis:', df.columns)
for var in df.columns:
    print('****************')
    print('Variável:', var)
    data = df[var]
    # define a distribuição teórica a ser testada
    dist = 'norm'
    # realiza o teste KS
    KS = kstest(data, dist)
    print('Valor p:',KS.pvalue)

Variáveis: Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')
****************
Variável: sepal length (cm)
Valor p: 0.0
****************
Variável: sepal width (cm)
Valor p: 1.9343513094431768e-253
****************
Variável: petal length (cm)
Valor p: 1.4044248603466367e-136
****************
Variável: petal width (cm)
Valor p: 1.8764992713715694e-42




---



Podemos ainda comparar duas distribuições de dados e verificar se vieram de um mesmo modelo.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
np.random.seed(2011)

# Mesmos parâmetros
mean = 5; std = 10
n = 1000
data1 = np.random.normal(mean,std,n)
data2 = np.random.normal(mean,std,n)

# realiza o teste KS de duas amostras
KS = ks_2samp(data1, data2)
print('Valor p:',KS.pvalue)

Valor p: 0.6101664688189142


In [ ]:
import numpy as np
from scipy.stats import ks_2samp
np.random.seed(2011)

# Médias e desvios diferentes
n = 1000
data1 = np.random.normal(10,2,n)
#data2 = np.random.normal(8,1,n)
data2 = np.random.exponential(10,n)

# realiza o teste KS de duas amostras
KS = ks_2samp(data1, data2)
print('Valor p:',KS.pvalue)

Valor p: 1.2279568309794965e-100


In [ ]:
import numpy as np
from scipy.stats import ks_2samp
np.random.seed(2011)

# gera os dados
mean = 5; std = 10
n = 1000
data1 = np.random.normal(mean,std,n)
data2 = np.random.exponential(mean,n)

# realiza o teste KS de duas amostras
KS = ks_2samp(data1, data2)
print('Valor p:',KS.pvalue)

Valor p: 1.866671893947957e-41


**Exemplo:** Verifique se o conjunto de dados a seguir segue um distribuição exponencial.

In [ ]:
import numpy as np
from scipy.stats import norm, kstest
np.random.seed(42)

# Amostra de dados
#lbd = 2.0
#n = 10
#data = expon.rvs(size=n)
#data = np.round(data*100)/100
#print('Dados:', data)
data =  [0.47, 3.01, 1.32, 0.91, 0.17,
         0.17, 0.06, 2.01, 0.92, 1.23]

# Definindo a distribuição teórica a ser testada
dist = 'expon'

# Realizando o teste KS
KS = kstest(data, dist)

# Imprimindo o resultado
print('Estatística de teste:', KS.statistic)
print('Valor p:',KS.pvalue)

Estatística de teste: 0.197475775966364
Valor p: 0.7616620889836793


In [ ]:
import numpy as np
from scipy.stats import norm, kstest
np.random.seed(42)

# Amostra de dados
data =  np.random.uniform(0,1,100)

# Definindo a distribuição teórica a ser testada
dist = 'expon'

# Realizando o teste KS
KS = kstest(data, dist)

# Imprimindo o resultado
print('Estatística de teste:', KS.statistic)
print('Valor p:',KS.pvalue)

Estatística de teste: 0.37273523519406004
Valor p: 5.584652200712474e-13


**Exemplo:** Usando o teste de Kolmogorov-Smirnov, determine se os dois conjuntos de dados a seguir vieram de uma mesma distribuição.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp
np.random.seed(100)

# Dados
#mean = 0
#std = 2
#n = 10
#data1 = np.random.normal(mean,std,n)
#data1 = np.round(data1*100)/100
#print('Dados:', data1)
#data2 = np.random.uniform(0,1,n)
#data2 = np.round(data2*100)/100
#print('Dados:', data2)

data1 = [0.99,-0.28,1.3,3.05,-0.47,
         -0.47,3.16,1.53,-0.94,1.09]
data2 = [0.18,0.18,0.3,0.52,0.43,
         0.29,0.61,0.14,0.29,0.37]

# realiza o teste KS de duas amostras
KS = ks_2samp(data1, data2)
print('Valor p:',KS.pvalue)

Valor p: 0.05244755244755244


**Exemplo:** Verifique se os atributos da flor íris tem a mesma distribuição.

In [ ]:
import pandas as pd
from sklearn import datasets

# carrega os dados
iris = datasets.load_iris()
# transforma em um dataframe do Pandas
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
print('Variáveis:', df.columns)
for var1 in df.columns:
  for var2 in df.columns:
    if(var1 != var2):
      print('****************')
      print(var1,' X ', var2)
      data1 = df[var1]
      data2 = df[var2]
      KS = ks_2samp(data1, data2)
      print('Valor p:',KS.pvalue)

Variáveis: Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')
****************
sepal length (cm)  X  sepal width (cm)
Valor p: 6.399337692587972e-87
****************
sepal length (cm)  X  petal length (cm)
Valor p: 5.290396633242105e-22
****************
sepal length (cm)  X  petal width (cm)
Valor p: 2.133112564195991e-89
****************
sepal width (cm)  X  sepal length (cm)
Valor p: 6.399337692587972e-87
****************
sepal width (cm)  X  petal length (cm)
Valor p: 4.092283006961527e-23
****************
sepal width (cm)  X  petal width (cm)
Valor p: 8.600928185744819e-66
****************
petal length (cm)  X  sepal length (cm)
Valor p: 5.290396633242105e-22
****************
petal length (cm)  X  sepal width (cm)
Valor p: 4.092283006961527e-23
****************
petal length (cm)  X  petal width (cm)
Valor p: 6.639808432803654e-32
****************
petal width (cm)  X  sepal length (cm)
Valor p: 2.133112564195991e-89